## Zork: The Unofficial Python Edition (Enhanced)

Welcome to the fully upgraded, notebook-safe version of our classic text adventure. 

This version runs on a scalable **State Machine** architecture and now features a persistent **Global Game State**. This means the engine actively tracks your inventory, remembers if you've opened containers, handles conditional locks (paths that require specific items to pass), and even supports basic combat encounters.

### 🎮 How to Play
Run the Python cell below to start the engine. You will interact with the game via the `input()` text box that appears below the cell. 

**Basic Commands:**
* **Movement:** `go southwest`, `go east`, `descend grating`
* **Actions:** `open mailbox`, `take machete`, `attack ogre`
* **Inventory:** 
  * Type `inventory` (or `i`) to see what you are carrying.
  * You can interact with items you hold (e.g., `read leaflet`).
* **Utility:** 
  * Type `look` (or `l`) to reprint your current surroundings.
  * Type `quit` to cleanly exit the game loop without crashing the notebook kernel.

*Your mission is to find the Jade Statue, but beware of what lurks in the clearing. Good luck!*

In [0]:
# ==========================================
# 1. THE GAME DATA (State & Map)
# ==========================================

game_state = {
    'inventory': [],
    'mailbox_open': False,
    'ogre_alive': True
}

rooms = {
    'field': {
        'desc': "You are standing in an open field west of a white house, with a boarded front door.\n(A secret path leads southwest into the forest.)\nThere is a Small Mailbox.",
        'exits': {'go southwest': 'forest'},
        'actions': {
            'take mailbox': "It is securely anchored.",
            'open mailbox': "__OPEN_MAILBOX__", 
            'take leaflet': "__TAKE_LEAFLET__", 
            'go east': "The door is boarded and you cannot remove the boards.",
            'open door': "The door cannot be opened.",
            'take boards': "The boards are securely fastened.",
            'look at house': "The house is a beautiful colonial house which is painted white. It is clear that the owners must have been extremely wealthy."
        }
    },
    'forest': {
        'desc': "This is a forest, with trees in all directions. To the east, there appears to be sunlight.\nThick vines block the path to the west.",
        'exits': {
            'go east': 'clearing',
            # Conditional Exit
            'go west': {
                'required_item': 'machete',
                'target_room': 'deep_forest',
                'fail_message': "You try to push through, but you need a machete to cut these thick vines."
            }
        },
        'actions': {
            'go north': "The forest becomes impenetrable to the North.",
            'go south': "Storm-tossed trees block your way."
        }
    },
    'deep_forest': {
        'desc': "You have hacked your way into the deep forest. It is eerily quiet here. A dead end.",
        'exits': {'go east': 'forest'},
        'actions': {}
    },
    'clearing': {
        'desc': "You are in a clearing. A massive ogre is blocking the path to the south.\nThere is an open grating, descending into darkness.\nA rusty machete is resting against a tree.",
        'exits': {'descend grating': 'cave'},
        'actions': {
            'take machete': "__TAKE_MACHETE__",
            'go south': "__GO_SOUTH__", 
            'attack ogre': "__ATTACK_OGRE__",
            'kill ogre': "__ATTACK_OGRE__"
        }
    },
    'ogre_lair': {
        'desc': "You have stepped past the defeated ogre into his lair. It smells terrible here, but you are safe.",
        'exits': {'go north': 'clearing'},
        'actions': {}
    },
    'cave': {
        'desc': "You are in a tiny cave with a dark, forbidding staircase leading down.\nThere is a skeleton of a human male in one corner.",
        'exits': {
            'descend staircase': 'mud_room',
            'go down staircase': 'mud_room',
            'scale staircase': 'mud_room'
        },
        'actions': {
            'take skeleton': "Why would you do that? Are you some sort of sicko?",
            'smash skeleton': "Sick person. Have some respect mate.",
            'light up room': "You would need a torch or lamp to do that.",
            'break skeleton': "I have two questions: Why and With What?",
            'suicide': "__DEATH__"
        }
    },
    'mud_room': {
        'desc': "You have entered a mud-floored room.\nLying half buried in the mud is an old trunk, bulging with jewels.",
        'exits': {},
        'actions': {
            'open trunk': "__WIN__" 
        }
    }
}


# ==========================================
# 2. THE GAME ENGINE (Logic)
# ==========================================

print("---------------------------------------------------------")
print("Welcome to Zork - The Unofficial Python Version.")
print("Commands: 'inventory', 'look', 'quit', or verb-noun actions (e.g. 'go east').")

current_room = 'field'
playing = True
just_entered_room = True

while playing:
    print("---------------------------------------------------------")
    
    # Print descriptions only when looking or entering a new room
    if just_entered_room:
        print(rooms[current_room]['desc'])
        # Dynamic additions based on state
        if current_room == 'field' and game_state['mailbox_open'] and 'leaflet' not in game_state['inventory']:
            print("There is a leaflet inside the open mailbox.")
        just_entered_room = False
        
    command = input("What do you do? ").lower().strip()
    if not command:
        continue

    room_data = rooms[current_room]
    
    # A. CHECK GLOBAL COMMANDS
    if command in ['inventory', 'i', 'inv']:
        if game_state['inventory']:
            print("You are carrying: " + ", ".join(game_state['inventory']))
        else:
            print("You are empty-handed.")
        continue
        
    elif command == 'read leaflet':
        if 'leaflet' in game_state['inventory']:
            print("Welcome to the Unofficial Python Version of Zork. Your mission is to find a Jade Statue.")
        elif current_room == 'field' and game_state['mailbox_open']:
            print("You read the leaflet in the mailbox: Welcome to the Unofficial Python Version of Zork...")
        else:
            print("You don't have a leaflet to read.")
        continue
            
    elif command in ['look', 'l']:
        just_entered_room = True
        continue
        
    elif command == 'quit':
        playing = False
        continue

    # B. CHECK MOVEMENT
    if command in room_data.get('exits', {}):
        exit_info = room_data['exits'][command]
        
        # Unlocked path (string)
        if isinstance(exit_info, str):
            current_room = exit_info
            just_entered_room = True
            
        # Locked path (dictionary)
        elif isinstance(exit_info, dict):
            if exit_info['required_item'] in game_state['inventory']:
                print(f"You use the {exit_info['required_item']} to clear the way!")
                current_room = exit_info['target_room']
                just_entered_room = True
            else:
                print(exit_info['fail_message'])
        continue

    # C. CHECK ROOM-SPECIFIC ACTIONS
    if command in room_data.get('actions', {}):
        response = room_data['actions'][command]
        
        # --- Custom Triggers ---
        if response == "__OPEN_MAILBOX__":
            if not game_state['mailbox_open']:
                game_state['mailbox_open'] = True
                print("Opening the small mailbox reveals a leaflet.")
            else:
                print("The mailbox is already open.")
                
        elif response == "__TAKE_LEAFLET__":
            if 'leaflet' in game_state['inventory']:
                print("You already have it.")
            elif game_state['mailbox_open']:
                game_state['inventory'].append('leaflet')
                print("You put the leaflet in your inventory.")
            else:
                print("You don't see a leaflet here.")

        elif response == "__TAKE_MACHETE__":
            if 'machete' in game_state['inventory']:
                print("You already have the machete.")
            else:
                game_state['inventory'].append('machete')
                print("You pick up the rusty machete. It's heavy but sharp.")
                # Update room desc so machete is gone, keeping the ogre status accurate
                ogre_text = "A massive ogre is blocking the path to the south." if game_state['ogre_alive'] else "The body of a defeated ogre lies on the ground.\nA path leads south."
                rooms['clearing']['desc'] = f"You are in a clearing. {ogre_text}\nThere is an open grating, descending into darkness."

        elif response == "__ATTACK_OGRE__":
            if not game_state['ogre_alive']:
                print("The ogre is already dead. Settle down.")
            elif 'machete' in game_state['inventory']:
                game_state['ogre_alive'] = False
                print("You swing the machete fiercely! The ogre falls to the ground, defeated. The southern path is clear.")
                # Update room desc so ogre is dead
                machete_text = "" if 'machete' in game_state['inventory'] else "\nA rusty machete is resting against a tree."
                rooms['clearing']['desc'] = f"You are in a clearing. The body of a defeated ogre lies on the ground.\nA path leads south, and an open grating descends into darkness.{machete_text}"
            else:
                print("You charge at the ogre with your bare hands. He laughs, picks you up, and eats you.")
                print("--- YOU DIED ---")
                if input("Do you want to respawn? Y/N ").lower() == 'y':
                    current_room = 'field'
                    just_entered_room = True
                else:
                    playing = False

        elif response == "__GO_SOUTH__":
            if game_state['ogre_alive']:
                print("The ogre growls and shoves you back. You cannot pass.")
            else:
                current_room = 'ogre_lair'
                just_entered_room = True
                
        elif response == "__DEATH__":
            print("You throw yourself down the staircase as an attempt at suicide. You die.")
            if input("Do you want to respawn? Y/N ").lower() == 'y':
                current_room = 'field'
                just_entered_room = True
            else:
                playing = False
                
        elif response == "__WIN__":
            print("You open the trunk and find the Jade Statue. You have completed your quest!")
            if input("Do you want to play again? Y/N ").lower() == 'y':
                # Quick reset for a new game
                current_room = 'field'
                game_state = {'inventory': [], 'mailbox_open': False, 'ogre_alive': True}
                just_entered_room = True
            else:
                playing = False
                
        else:
            # Print standard text responses
            print(response)
            
    else:
        # Failsafe for unrecognized commands
        print("I don't understand that, or you can't do that here.")

print("Thanks for playing! The kernel is now safely resting.")